In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/SIGAIDA/tra_acc.csv"
df = pd.read_csv(path)
df.head()

/tmp/ipykernel_334/1860369094.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,797bf29824ec18668debca66975762eea5bedc02195334...,NaN,03/08/2026 11:47:00 PM,35,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,NOT DIVIDED,...,3.0,0.0,1.0,0.0,23,1,3,41.906250,-87.726557,POINT (-87.726556773589 41.906250266381)
1,280d6a1651dde4eb4ade75e2b50b3855eccfd192cdec5e...,NaN,03/08/2026 11:15:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,ONE-WAY,...,0.0,0.0,3.0,0.0,23,1,3,41.893081,-87.633946,POINT (-87.63394607418 41.893080981391)
2,efe22e22c5d01db6bf35459d0390e1ff6a5b1673a03695...,NaN,03/08/2026 11:08:00 PM,25,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,ONE-WAY,...,1.0,1.0,2.0,0.0,23,1,3,41.781053,-87.693218,POINT (-87.693218110419 41.781053217434)
3,9c032b1880ccd0784ebb8aac17b7745a7801ec3187f9d9...,NaN,03/08/2026 11:05:00 PM,25,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,...,0.0,0.0,2.0,0.0,23,1,3,NaN,NaN,NaN
4,6aea8b3016c4b0ac5d5135b2f94af6cc05be3d78d45cde...,NaN,03/08/2026 10:45:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,"DARKNESS, LIGHTED ROAD",SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),...,1.0,1.0,2.0,0.0,22,1,3,41.774965,-87.722812,POINT (-87.722812226815 41.774964998435)


In [ ]:
import torch

In [3]:
import sklearn

In [4]:
df['CRASH_DATE'] = pd.to_datetime(df['CRASH_DATE'])
df['CRASH_HOUR']        = df['CRASH_DATE'].dt.hour
df['CRASH_DAY_OF_WEEK'] = df['CRASH_DATE'].dt.dayofweek  # 0=Monday, 6=Sunday
df['CRASH_MONTH']       = df['CRASH_DATE'].dt.month

/tmp/ipykernel_334/2455376617.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['CRASH_DATE'] = pd.to_datetime(df['CRASH_DATE'])


In [6]:
df1 = df[[
    'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT',
    'ALIGNMENT', 'TRAFFICWAY_TYPE', 'LANE_CNT', 'POSTED_SPEED_LIMIT',
    'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'INTERSECTION_RELATED_I',
    'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
    'FIRST_CRASH_TYPE'
]].copy()

df1.dropna(inplace=True)

df1['LANE_CNT'] = pd.to_numeric(df1['LANE_CNT'], errors='coerce')
df1.dropna(subset=['LANE_CNT'], inplace=True)
df1['LANE_CNT'] = df1['LANE_CNT'].astype(int)

df1_categorical_cols = [
    'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT',
    'ALIGNMENT', 'TRAFFICWAY_TYPE', 'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION',
    'FIRST_CRASH_TYPE'
]
for col in df1_categorical_cols:
    df1 = df1[df1[col].str.upper().str.strip() != 'UNKNOWN']

for col in df1_categorical_cols:
    df1 = df1[df1[col].str.strip() != '']

df1 = df1[df1['INTERSECTION_RELATED_I'].str.upper().str.strip().isin(['Y', 'N'])]

df1 = df1[df1['POSTED_SPEED_LIMIT'] > 0]           # no zero/negative speed limits
df1 = df1[df1['POSTED_SPEED_LIMIT'] <= 100]        # no absurd speed limits
df1 = df1[df1['LANE_CNT'] > 0]                     # must have at least 1 lane
df1 = df1[df1['LANE_CNT'] <= 20]                   # sanity cap on lane count
df1 = df1[df1['CRASH_HOUR'].between(0, 23)]
df1 = df1[df1['CRASH_DAY_OF_WEEK'].between(0, 6)]
df1 = df1[df1['CRASH_MONTH'].between(1, 12)]

df1.reset_index(drop=True, inplace=True)

In [7]:
df2 = df[[
    'FIRST_CRASH_TYPE', 'CRASH_TYPE', 'WEATHER_CONDITION', 'LIGHTING_CONDITION',
    'ROADWAY_SURFACE_COND', 'POSTED_SPEED_LIMIT', 'TRAFFICWAY_TYPE',
    'INTERSECTION_RELATED_I', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'DAMAGE', 'NUM_UNITS',
    'HIT_AND_RUN_I'
]].copy()

df2.dropna(inplace=True)

df2_categorical_cols = [
    'FIRST_CRASH_TYPE', 'CRASH_TYPE', 'WEATHER_CONDITION', 'LIGHTING_CONDITION',
    'ROADWAY_SURFACE_COND', 'TRAFFICWAY_TYPE'
]
for col in df2_categorical_cols:
    df2 = df2[df2[col].str.upper().str.strip() != 'UNKNOWN']

for col in df2_categorical_cols:
    df2 = df2[df2[col].str.strip() != '']

df2 = df2[df2['INTERSECTION_RELATED_I'].str.upper().str.strip().isin(['Y', 'N'])]

df2 = df2[df2['HIT_AND_RUN_I'].str.upper().str.strip().isin(['Y', 'N'])]

valid_damage_values = ['$500 OR LESS', '$501 - $1,500', 'OVER $1,500']
df2 = df2[df2['DAMAGE'].str.upper().str.strip().isin(valid_damage_values)]

df2 = df2[df2['POSTED_SPEED_LIMIT'] > 0]
df2 = df2[df2['POSTED_SPEED_LIMIT'] <= 100]
df2 = df2[df2['NUM_UNITS'] > 0]                    # must involve at least 1 unit
df2 = df2[df2['NUM_UNITS'] <= 50]                  # sanity cap
df2 = df2[df2['CRASH_HOUR'].between(0, 23)]
df2 = df2[df2['CRASH_DAY_OF_WEEK'].between(0, 6)]

df2.reset_index(drop=True, inplace=True)